# Overfitting test of the prior training

A sanity check of the training pipeline (suggested by Florian): a model must fit data it was trained on better than data it has not seen.

Take `N_BATCHES` fixed batches of the training data. All experiments validate on these same N batches:

| | trained on | validated on | expected |
|---|---|---|---|
| **E1** | all N batches | the same N batches | high log-likelihood that keeps rising: the model memorizes them |
| **E2** | only the first `K_BATCHES` (1-2) of them | the N batches | lower than E1: most of the N batches were never seen |

**E1 must be better than E2 on the N batches. If not, there is a bug.**

E2 is also split up: its log-likelihood on the K batches it was trained on (should be close to E1 or higher) and on the other N - K (should be clearly lower, and may fall with training: overfitting).

What is tested: the real pipeline of `SO3PriorTrainer` (model, optimizer, LR scheduler, gradient clipping, `train()`, `validate()`) with the settings of the pretrained prior `id_0900`. Only the datasets are replaced by the fixed batches (via `AMASS(data=...)`).

Two more checks:
- the validation loss of the pipeline equals the log-likelihood computed directly here, on the same data;
- no log-likelihood is NaN (`forward_kld` uses `nanmean`, which would silently ignore NaNs).

The batches are poses drawn at random from the whole training split of the compressed AMASS (every 10th frame), not consecutive frames: consecutive frames are almost identical and would leak from the K batches into the others.

In [ ]:
import importlib
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from hp.core.prior_training import make_config
from hp.data.amass import AMASS
from hp.data.amass_cfg import ORIGINAL_AMASS_SPLITS
from hp.utils.paths import REPO_ROOT

# ---- parameters ----------------------------------------------------------------------
BASE_CONFIG = REPO_ROOT / "experiments" / "pretrained_models" / "id_0900" / "config.yaml"  # model and training settings
ROTATIONS = "~/data/AMASS_rotations_c10"   # rotation files of the compressed AMASS; the batches come from its train split
OUT_DIR = "~/data/experiments/overfit_test"  # results (json, plot); no model checkpoints are saved
N_BATCHES = 10            # the fixed batches, also the validation data of every experiment
K_BATCHES = 2             # E2 trains on the first K of them
BATCH_SIZE = 2000         # as id_0900
MAX_ITERATIONS = 1000     # per experiment; about 3.3 s per iteration on an M5 Pro CPU -> ~1 h each
VALID_INTERVAL = 50       # iterations between two validations
DEVICE = "cpu"
SEED = 10                 # model initialization and training, the same in all experiments
DATA_SEED = 0             # choice of the batches
QUICK = False             # True: tiny run (batch 64, 40 iterations) to check that the notebook works
# ----------------------------------------------------------------------------------------

if QUICK:
    BATCH_SIZE, MAX_ITERATIONS, VALID_INTERVAL = 64, 40, 10

ROTATIONS, OUT_DIR = Path(ROTATIONS).expanduser(), Path(OUT_DIR).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HUPROSO_AMASS_ROTATIONS"] = str(ROTATIONS)  # this kernel only
os.environ["HUPROSO_AMASS_SPLIT"] = "original"
os.environ["HUPROSO_DEVICE"] = DEVICE
os.environ.pop("HUPROSO_SUBSET", None)
print(f"{N_BATCHES} batches of {BATCH_SIZE}, E2 trains on {K_BATCHES}; {MAX_ITERATIONS} iterations, validation every {VALID_INTERVAL}")

## The N batches

Random poses of the training split (official AMASS split), without repetition. Batch k is rows `k * BATCH_SIZE ... (k + 1) * BATCH_SIZE`.

In [ ]:
names = ORIGINAL_AMASS_SPLITS["train"]
parts = [torch.load(ROTATIONS / f"{n}.pt")[:, :22].float() for n in names]  # quaternions [x, y, z, w], as AMASS.load_data
sizes = np.array([len(p) for p in parts])
pool = torch.cat(parts)
del parts

rng = np.random.default_rng(DATA_SEED)
picked = rng.choice(len(pool), N_BATCHES * BATCH_SIZE, replace=False)
batches = AMASS(data=pool[picked])  # rotation matrices [N * BATCH_SIZE, 19, 3, 3], batch k = rows k*B ... (k+1)*B
trained_rows = np.arange(K_BATCHES * BATCH_SIZE)  # E2 trains on these
other_rows = np.arange(K_BATCHES * BATCH_SIZE, N_BATCHES * BATCH_SIZE)

print(f"training split: {len(pool):,} poses from {len(names)} datasets; picked {len(picked):,}")
per_dataset = np.bincount(np.searchsorted(np.cumsum(sizes), picked, side="right"), minlength=len(names))
print("picked per dataset:", ", ".join(f"{n} {c}" for n, c in zip(names, per_dataset)))

## Training with the pipeline



`run(train_rows)` builds an `SO3PriorTrainer` from the config of `id_0900` (same seed, so the same initial model in every experiment), replaces its training data by `train_rows` of the batches and its validation data by all N batches, and calls its `train()`. At every validation it also records, computed directly here:
- the log-likelihood on all N batches, on the K batches E2 trains on, and on the other N - K;
- the number of NaN log-likelihoods.

Log-likelihoods are per pose (higher is better); the pipeline reports the loss = minus the log-likelihood.

In [ ]:
def log_likelihoods(trainer, rows):
    """Log-likelihood of every pose in rows of the batches, without gradients, in batches of BATCH_SIZE."""
    trainer.pdf_estimator.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(rows), BATCH_SIZE):
            x = batches.data[rows[start : start + BATCH_SIZE]].to(DEVICE)
            out.append(trainer.forward_estimator(x).cpu())
    trainer.pdf_estimator.train()
    return torch.cat(out).double().numpy()


def run(train_rows):
    cfg = make_config(BASE_CONFIG, exp_id=0, save_root=str(OUT_DIR), max_iterations=MAX_ITERATIONS,
                      valid_interval=VALID_INTERVAL, valid_wasserstein=False, device=DEVICE, seed=SEED,
                      save_ckpt_every=10**9,  # no checkpoints
                      overrides={"batch_size": BATCH_SIZE, "only_eval": True, "evaluate": False,
                                 "so3.condition": False, "conditioning.conditioning_modality": "", "valid.inv_kin": False})
    np.random.seed(SEED), torch.manual_seed(SEED)
    trainer = getattr(importlib.import_module("hp.core.trainer"), cfg.trainer)(cfg)  # only_eval: loads no data
    trainer.only_eval = False
    trainer.dataset = trainer.train_dataset = torch.utils.data.Subset(batches, train_rows)
    trainer.val_dataset = batches
    trainer.create_optimizer_and_scheduler()
    trainer.create_dataloaders()
    assert trainer.max_iter == MAX_ITERATIONS, f"the scheduler caps the training at {trainer.max_iter} iterations"

    history = []
    pipeline_validate = trainer.validate

    def validate():
        metrics = pipeline_validate()
        ll = log_likelihoods(trainer, np.arange(len(batches)))
        row = dict(iteration=(len(history) + 1) * VALID_INTERVAL,
                   pipeline_valid_ll=-float(metrics["nll"]),
                   ll_all=float(np.nanmean(ll)),
                   ll_trained=float(np.nanmean(ll[trained_rows])),
                   ll_other=float(np.nanmean(ll[other_rows])),
                   nan=int(np.isnan(ll).sum()))
        history.append(row)
        print(f"it {row['iteration']:5d} | LL all {row['ll_all']:8.2f} | first {K_BATCHES} batches {row['ll_trained']:8.2f}"
              f" | other {N_BATCHES - K_BATCHES} {row['ll_other']:8.2f} | pipeline valid {row['pipeline_valid_ll']:8.2f} | NaN {row['nan']}")
        return metrics

    trainer.validate = validate
    start = time.time()
    train_loss, _ = trainer.train()
    return dict(train_rows=len(train_rows), history=history, train_ll=(-train_loss).tolist(), minutes=(time.time() - start) / 60)


results = {}

## E1: train on all N batches

In [ ]:
results["E1"] = run(np.arange(len(batches)))

## E2: train on the first K batches

In [ ]:
results["E2"] = run(trained_rows)

## Result

In [ ]:
e1, e2 = results["E1"]["history"], results["E2"]["history"]
it = [r["iteration"] for r in e1]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(it, [r["ll_all"] for r in e1], color="C0", label=f"E1 (trained on all {N_BATCHES}): all {N_BATCHES} batches")
ax.plot(it, [r["ll_all"] for r in e2], color="C1", label=f"E2 (trained on {K_BATCHES}): all {N_BATCHES} batches")
ax.plot(it, [r["ll_trained"] for r in e2], color="C1", ls=":", label=f"E2: its {K_BATCHES} training batches")
ax.plot(it, [r["ll_other"] for r in e2], color="C1", ls="--", label=f"E2: the other {N_BATCHES - K_BATCHES} batches (never seen)")
ax.set_xlabel("iteration")
ax.set_ylabel("log-likelihood per pose")# (higher is better)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "overfit_test.png", dpi=120)
plt.show()

last1, last2 = e1[-1], e2[-1]
print(f"after {last1['iteration']} iterations, log-likelihood on the {N_BATCHES} batches:")
print(f"  E1 (trained on all {N_BATCHES}):  {last1['ll_all']:8.2f}")
print(f"  E2 (trained on {K_BATCHES}):       {last2['ll_all']:8.2f}  (its {K_BATCHES} batches {last2['ll_trained']:.2f}, the other {N_BATCHES - K_BATCHES} {last2['ll_other']:.2f})")

checks = {
    "E1 > E2 on the N batches": last1["ll_all"] > last2["ll_all"],
    "E2: its training batches > the other batches": last2["ll_trained"] > last2["ll_other"],
    "pipeline validation = direct computation": all(abs(r["pipeline_valid_ll"] - r["ll_all"]) < 1e-3 * max(1, abs(r["ll_all"])) for r in e1 + e2),
    "no NaN log-likelihood": all(r["nan"] == 0 for r in e1 + e2),
}
for name, ok in checks.items():
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
if not checks["E1 > E2 on the N batches"]:
    print("E1 is not better than E2: probably a bug in the training or validation.")

with open(OUT_DIR / "overfit_test.json", "w") as f:
    json.dump(dict(params=dict(n_batches=N_BATCHES, k_batches=K_BATCHES, batch_size=BATCH_SIZE, max_iterations=MAX_ITERATIONS,
                               valid_interval=VALID_INTERVAL, seed=SEED, data_seed=DATA_SEED, rotations=str(ROTATIONS)),
                   checks=checks, results=results), f, indent=1)
print("saved to", OUT_DIR)

In [ ]:
щ